In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import datetime

# 불필요한 경고문 생략(선택)
import warnings
warnings.filterwarnings('ignore')

# 모든 컬럼 출력설정(선택)
pd.set_option('display.max_columns', None)

#데이터 불러오기 
df = pd.read_csv('total_data.csv',index_col=0)

print('[행/컬럼 갯수]')
print(f"행: {df.shape[0]}, 컬럼: {df.shape[1]}\n")

[행/컬럼 갯수]
행: 51279, 컬럼: 40



In [2]:
#주차 컬럼 날짜타입 변환 (범주->날짜형)
df['주차'] = pd.to_datetime(df['주차'], format='%Y%m%d')
df['주차'].info()

<class 'pandas.core.series.Series'>
Index: 51279 entries, 0 to 51278
Series name: 주차
Non-Null Count  Dtype         
--------------  -----         
51279 non-null  datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 801.2 KB


In [3]:
# 결측치 확인 -> 없음 
df.isna().sum()

기간        0
주차        0
라인        0
성별        0
기획년도      0
시즌이월      0
상품년차      0
시즌        0
복종        0
소품종       0
CAT       0
총입고수량     0
총입고원가     0
총입고택가     0
총출고수량     0
총출고원가     0
총출고택가     0
판매액       0
판매수량      0
매출원가      0
판매택가      0
총판매액      0
총판매수량     0
총매출원가     0
총판매택가     0
물류재고수량    0
물류재고원가    0
물류재고택가    0
매장재고수량    0
매장재고원가    0
매장재고택가    0
재고수량      0
재고원가      0
재고택가      0
기간입고수량    0
기간입고원가    0
기간입고택가    0
기간출고수량    0
기간출고원가    0
기간출고택가    0
dtype: int64

In [4]:
# 중복값 확인 및 제거 -> 전체 중복 12개
df.duplicated().sum() 
df.drop_duplicates(inplace=True)
# df[df.duplicated(keep=False)].sort_values(by='주차')

print('[행/컬럼 갯수]')
print(f"행: {df.shape[0]}, 컬럼: {df.shape[1]}\n")

[행/컬럼 갯수]
행: 51267, 컬럼: 40



# 범주형 컬럼 확인

In [5]:
#성별 소품종 클래스 확인 : 기타로 분류되는 클래스 2개 존재
#--> 최종 '기타' 로 오분류된 항목 52개 변환
display(df['성별'].value_counts())

sex_df = df[df['성별'].str.contains('기타')]
sex_df['소품종'].value_counts()

#봄 패딩 베스트만, '기타' 로 분류됨 -> 성별 구분 착오 예상 --> 봄패딩베스트 '1:남성' 값으로 변환 
con = (df['소품종']=='패딩베스트') & (df['시즌']=='봄')
df.loc[con,'성별'] = '1:남성'

df.loc[con,'성별'].value_counts()

성별
1:남성      47346
3:남녀공용     2510
2:여성       1322
4:기타         89
Name: count, dtype: int64

성별
1:남성    52
Name: count, dtype: int64

- Q. 시즌 이월 확인 코드?

In [6]:
#시즌이월 컬럼 클래스 확인 : 이월제품 의미 파악 필요
##결론 : 판매예측/할인최적화 모델링시 '이월' 행 삭제 ( -12578 32%) ,년간 매출 집계시 유지
display(df['시즌이월'].value_counts())

#2024년도 기준 시즌/이월 여부 확인 
df_2024 = df[df['기획년도']==2024]
df_2024 = df_2024.drop(columns=['기간','상품년차'])

# # 범주형 최소단위 필터링을 위한 '카테고리' 컬럼 생성
df_2024['카테고리'] = df_2024['시즌'] + "_" +  df_2024['복종'] + "_" + df_2024['소품종'] + "_" + df_2024['라인']+ "_" + df_2024['성별']

# #카테고리별 시즌/이월값이 둘다 있는거 소팅 -> '샘플 가을_니트 셔츠_라운드_ZB_1:남성'   확인
df_2024.groupby('카테고리')['시즌이월'].nunique()

# #샘플확인 : '가을_니트 셔츠_라운드_ZB_1:남성' -> 시즌별 마감 이후 이월로 변경 됨 
con = df_2024['카테고리'] == '가을_니트 셔츠_라운드_ZB_1:남성'
smpl = df_2024[con].sort_values(by='주차',ascending=True)
smpl[(smpl['주차'] >'2024-11-01') & (smpl['주차'] <='2024-12-30')]

# 시즌-> 이월 바뀌는 시점 함수화 : gpt
def find_transition_points(group):
    group = group.sort_values('주차')
    transition_rows = group[(group['시즌이월'].shift(1) == '01_시즌') & (group['시즌이월'] == '02_이월')]
    return transition_rows[['카테고리', '주차']]

transition_points = df_2024.groupby('카테고리', group_keys=False).apply(find_transition_points)

# 시즌 -> 이월로 바뀌는 주차만 출력
transition_points['주차'].unique()

시즌이월
01_시즌    38689
02_이월    12578
Name: count, dtype: int64

<DatetimeArray>
['2024-12-01 00:00:00', '2024-06-02 00:00:00', '2024-10-06 00:00:00']
Length: 3, dtype: datetime64[ns]

In [7]:
print('[행/컬럼 갯수]')
print(f"행: {df.shape[0]}, 컬럼: {df.shape[1]}\n")

df.head(3)

[행/컬럼 갯수]
행: 51267, 컬럼: 40



,기간,주차,라인,성별,기획년도,시즌이월,상품년차,시즌,복종,소품종,CAT,총입고수량,총입고원가,총입고택가,총출고수량,총출고원가,총출고택가,판매액,판매수량,매출원가,판매택가,총판매액,총판매수량,총매출원가,총판매택가,물류재고수량,물류재고원가,물류재고택가,매장재고수량,매장재고원가,매장재고택가,재고수량,재고원가,재고택가,기간입고수량,기간입고원가,기간입고택가,기간출고수량,기간출고원가,기간출고택가
0,당해,2021-01-03,ZB,1:남성,2021,01_시즌,당해년도,봄,우븐 셔츠,캐쥬얼셔츠,02_SHIRTS,2247,19893640,157065300,1009,8933103,70529100,-124850,0,0,0,84850,3,26560,209700,1238,10960537,86536200,1006,8906543,70319400,2244,19867080,156855600,0,0,0,53,469228,3704700
1,당해,2021-01-03,ZB,1:남성,2021,01_시즌,당해년도,사계절,소품,양말,10_ACC/ETC,14000,11480000,46200000,10775,8835500,35557500,2033156,617,505940,2036100,10270838,3127,2564140,10319100,3225,2644500,10642500,7648,6271360,25238400,10873,8915860,35880900,0,0,0,454,372280,1498200
2,당해,2021-01-03,ZB,1:남성,2021,01_시즌,당해년도,봄,니트 셔츠,라운드,01_KNIT,20076,178239296,1403312400,5940,52736683,415206000,18981424,401,3559559,28029900,18981424,401,3560170,28029900,14136,125502613,988106400,5539,49176513,387176100,19675,174679125,1375282500,0,0,0,5940,52713656,415206000


In [8]:
df['성별'].unique()

array(['1:남성', '4:기타', '3:남녀공용', '2:여성'], dtype=object)

In [9]:
df[df['성별'] == '2:여성']

,기간,주차,라인,성별,기획년도,시즌이월,상품년차,시즌,복종,소품종,CAT,총입고수량,총입고원가,총입고택가,총출고수량,총출고원가,총출고택가,판매액,판매수량,매출원가,판매택가,총판매액,총판매수량,총매출원가,총판매택가,물류재고수량,물류재고원가,물류재고택가,매장재고수량,매장재고원가,매장재고택가,재고수량,재고원가,재고택가,기간입고수량,기간입고원가,기간입고택가,기간출고수량,기간출고원가,기간출고택가
18642,당해,2022-09-11,ZB,2:여성,2022,01_시즌,당해년도,사계절,소품,양말,10_ACC/ETC,5500,6600000,32450000,0,0,0,0,0,0,0,0,0,0,0,5500,6600000,32450000,0,0,0,5500,6600000,32450000,5500,6600000,32450000,0,0,0
18926,당해,2022-09-18,ZB,2:여성,2022,01_시즌,당해년도,사계절,소품,양말,10_ACC/ETC,5500,6600000,32450000,0,0,0,0,0,0,0,0,0,0,0,5500,6600000,32450000,0,0,0,5500,6600000,32450000,0,0,0,0,0,0
19382,당해,2022-09-25,ZB,2:여성,2022,01_시즌,당해년도,사계절,소품,양말,10_ACC/ETC,5500,6600000,32450000,0,0,0,0,0,0,0,0,0,0,0,5500,6600000,32450000,0,0,0,5500,6600000,32450000,0,0,0,0,0,0
19474,당해,2022-10-02,ZD,2:여성,2022,01_시즌,당해년도,겨울,스웨터,라운드,03_SWEATER,197,4373189,31500300,0,0,0,0,0,0,0,0,0,0,0,197,4373189,31500300,0,0,0,197,4373189,31500300,197,3415389,31500300,0,0,0
19521,당해,2022-10-02,ZD,2:여성,2022,01_시즌,당해년도,겨울,팬츠,팬츠(일반),04_PANTS,199,5723208,35621000,0,0,0,0,0,0,0,0,0,0,0,199,5723208,35621000,0,0,0,199,5723208,35621000,199,5723208,35621000,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51250,당해,2024-12-29,ZF,2:여성,2024,02_이월,당해년도,여름,스웨터,T-에리,03_SWEATER,622,8971540,61578000,143,2062589,14157000,0,0,0,0,7182253,143,2062589,14157000,475,6851256,47025000,0,0,0,475,6851256,47025000,0,0,0,0,0,0
51252,당해,2024-12-29,ZF,2:여성,2024,02_이월,당해년도,여름,팬츠,팬츠(일반),04_PANTS,494,6799075,48906000,104,1431384,10296000,0,0,0,0,5311817,101,1390094,9999000,389,5353927,38511000,3,41290,297000,392,5395217,38808000,0,0,0,0,0,0
51256,당해,2024-12-29,ZF,2:여성,2024,02_이월,당해년도,여름,우븐 셔츠,캐쥬얼셔츠,02_SHIRTS,273,5214300,27027000,83,1585300,8217000,0,0,0,0,5216588,83,1585300,8217000,187,3571700,18513000,0,0,0,187,3571700,18513000,0,0,0,0,0,0
51262,당해,2024-12-29,ZF,2:여성,2024,01_시즌,당해년도,겨울,자켓,싱글재킷,06_OUTER,208,8699497,41392000,121,5060765,24079000,199000,2,83649,398000,1253700,10,418245,1990000,87,3638732,17313000,111,4642520,22089000,198,8281252,39402000,0,0,0,0,0,0


# 수치형 컬럼 & 집계 컬럼 점검
사용 컬럼 : '총입고수량','총입고원가','총입고택가','총출고수량','총출고원가','총출고택가','판매수량','판매액','매출원가','판매택가','총판매액','총판매수량','총매출원가','총판매택가','재고수량','재고원가','재고택가'

재언 : 사용 컬럼에 성별이 없어서 추가했어요

In [10]:
# 총입고수량/입고원가/입고택가  -> 전처리 (-162행)
# 입고전 데이터 확인 및 행 삭제 : 162개 -> 입고되지 않은 상품은 출고 및 판매 불가, 예약판매 등 특수한 케이스 없다고 가정 

#1. 범주형 최소단위 필터링을 위한 '카테고리' 컬럼 생성
df['카테고리'] = df['시즌'] + "_" +  df['복종'] + "_" + df['소품종'] + "_" + df['라인']
num_df = df[['주차','라인', '성별','기획년도','시즌이월','시즌','복종','소품종', '카테고리', '총입고수량','총입고원가','총입고택가','판매수량','판매액','매출원가','판매택가','총판매액','총판매수량','총매출원가','총판매택가','재고수량','재고원가','재고택가']]

print((num_df[f'총입고수량'] == 0).sum())
filtered_df = num_df[(df['총입고수량'] > 0)]

print('[행/컬럼 갯수]')
print(f"행: {filtered_df.shape[0]}, 컬럼: {filtered_df.shape[1]}\n")

162
[행/컬럼 갯수]
행: 51105, 컬럼: 23



In [11]:
# 재고수량/재고원가/재고택가 정합성 확인 -> 재고 관련 컬럼 삭제 
# 결론: 재고관련 집계 컬럼 삭제 후 입고-판매 기준 다시 집계 (입고,판매 데이터의 신뢰도가 더 높다고 봄, 실제 wms랑 비교 할 수 없으므로 가정)
filtered_df['재고잔량_check'] = (filtered_df['총입고수량'] - filtered_df['총판매수량'] == filtered_df['재고수량'])
filtered_df['재고원가_check'] = (filtered_df['총입고원가'] - filtered_df['총매출원가'] == filtered_df['재고원가'])
filtered_df['재고택가_check'] = (filtered_df['총입고택가'] - filtered_df['총판매택가'] == filtered_df['재고원가'])

# 입고 - 판매 = 재고 안맞는 행 : 27569행
filtered_df[filtered_df[['재고잔량_check', '재고원가_check', '재고택가_check']].any(axis=1) == False]
(filtered_df[['재고잔량_check', '재고원가_check', '재고택가_check']].any(axis=1) == False).sum() #27569행

# 재고관련 컬럼 삭제
filtered_df.drop(columns=['재고수량','재고원가','재고택가','재고잔량_check','재고원가_check','재고택가_check'],inplace=True)

print('[행/컬럼 갯수]')
print(f"행: {filtered_df.shape[0]}, 컬럼: {filtered_df.shape[1]}\n")

[행/컬럼 갯수]
행: 51105, 컬럼: 20



# 여기서부터 재언

### 확인

In [12]:
filtered_df

,주차,라인,성별,기획년도,시즌이월,시즌,복종,소품종,카테고리,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,총판매액,총판매수량,총매출원가,총판매택가
0,2021-01-03,ZB,1:남성,2021,01_시즌,봄,우븐 셔츠,캐쥬얼셔츠,봄_우븐 셔츠_캐쥬얼셔츠_ZB,2247,19893640,157065300,0,-124850,0,0,84850,3,26560,209700
1,2021-01-03,ZB,1:남성,2021,01_시즌,사계절,소품,양말,사계절_소품_양말_ZB,14000,11480000,46200000,617,2033156,505940,2036100,10270838,3127,2564140,10319100
2,2021-01-03,ZB,1:남성,2021,01_시즌,봄,니트 셔츠,라운드,봄_니트 셔츠_라운드_ZB,20076,178239296,1403312400,401,18981424,3559559,28029900,18981424,401,3560170,28029900
3,2021-01-03,ZA,1:남성,2021,01_시즌,사계절,소품,타이,사계절_소품_타이_ZA,7652,49967560,381834800,174,8450445,1136220,8682600,85204225,1733,11316490,86476700
4,2021-01-03,ZA,1:남성,2021,01_시즌,봄,수트,블레이져(수트),봄_수트_블레이져(수트)_ZA,405,26409229,153495000,6,1682760,391248,2274000,16797003,60,3912478,22740000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51274,2024-12-29,ZA,1:남성,2024,01_시즌,사계절,소품,타이,사계절_소품_타이_ZA,1504,11731200,73696000,4,196000,31200,196000,38217400,789,6154200,38661000
51275,2024-12-29,ZB,1:남성,2024,01_시즌,겨울,수트,수트팬츠,겨울_수트_수트팬츠_ZB,1404,38437753,209196000,25,2227550,684433,3725000,28603530,272,7446630,40528000
51276,2024-12-29,ZB,1:남성,2024,02_이월,여름,스웨터,라운드,여름_스웨터_라운드_ZB,1926,28828072,190674000,0,0,0,0,56664950,1241,18575097,122859000
51277,2024-12-29,ZE,1:남성,2024,01_시즌,사계절,우븐 셔츠,드레스셔츠,사계절_우븐 셔츠_드레스셔츠_ZE,19614,139893880,1567158600,0,0,0,0,302854418,19428,138567263,1552297200


In [13]:
filtered_df['라인'].unique()

array(['ZB', 'ZA', 'ZE', 'ZD', 'ZC', 'ZF', 'ZG'], dtype=object)

In [14]:
filtered_df['성별'].unique()

array(['1:남성', '4:기타', '3:남녀공용', '2:여성'], dtype=object)

## 성별 == 여성 제외

In [15]:
using_df = filtered_df[~(filtered_df['성별'] == '2:여성')]
using_df

,주차,라인,성별,기획년도,시즌이월,시즌,복종,소품종,카테고리,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,총판매액,총판매수량,총매출원가,총판매택가
0,2021-01-03,ZB,1:남성,2021,01_시즌,봄,우븐 셔츠,캐쥬얼셔츠,봄_우븐 셔츠_캐쥬얼셔츠_ZB,2247,19893640,157065300,0,-124850,0,0,84850,3,26560,209700
1,2021-01-03,ZB,1:남성,2021,01_시즌,사계절,소품,양말,사계절_소품_양말_ZB,14000,11480000,46200000,617,2033156,505940,2036100,10270838,3127,2564140,10319100
2,2021-01-03,ZB,1:남성,2021,01_시즌,봄,니트 셔츠,라운드,봄_니트 셔츠_라운드_ZB,20076,178239296,1403312400,401,18981424,3559559,28029900,18981424,401,3560170,28029900
3,2021-01-03,ZA,1:남성,2021,01_시즌,사계절,소품,타이,사계절_소품_타이_ZA,7652,49967560,381834800,174,8450445,1136220,8682600,85204225,1733,11316490,86476700
4,2021-01-03,ZA,1:남성,2021,01_시즌,봄,수트,블레이져(수트),봄_수트_블레이져(수트)_ZA,405,26409229,153495000,6,1682760,391248,2274000,16797003,60,3912478,22740000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51274,2024-12-29,ZA,1:남성,2024,01_시즌,사계절,소품,타이,사계절_소품_타이_ZA,1504,11731200,73696000,4,196000,31200,196000,38217400,789,6154200,38661000
51275,2024-12-29,ZB,1:남성,2024,01_시즌,겨울,수트,수트팬츠,겨울_수트_수트팬츠_ZB,1404,38437753,209196000,25,2227550,684433,3725000,28603530,272,7446630,40528000
51276,2024-12-29,ZB,1:남성,2024,02_이월,여름,스웨터,라운드,여름_스웨터_라운드_ZB,1926,28828072,190674000,0,0,0,0,56664950,1241,18575097,122859000
51277,2024-12-29,ZE,1:남성,2024,01_시즌,사계절,우븐 셔츠,드레스셔츠,사계절_우븐 셔츠_드레스셔츠_ZE,19614,139893880,1567158600,0,0,0,0,302854418,19428,138567263,1552297200


### 확인

In [16]:
using_df['성별'].unique()

array(['1:남성', '4:기타', '3:남녀공용'], dtype=object)

## 사용할 컬럼만 다시 추출

In [17]:
using_df = using_df[['라인', '성별', '기획년도', '시즌이월', '시즌', '복종', '소품종', '카테고리', '주차', '총입고수량', '총입고원가', '총입고택가', '판매수량', '판매액']]
using_df

,라인,성별,기획년도,시즌이월,시즌,복종,소품종,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액
0,ZB,1:남성,2021,01_시즌,봄,우븐 셔츠,캐쥬얼셔츠,봄_우븐 셔츠_캐쥬얼셔츠_ZB,2021-01-03,2247,19893640,157065300,0,-124850
1,ZB,1:남성,2021,01_시즌,사계절,소품,양말,사계절_소품_양말_ZB,2021-01-03,14000,11480000,46200000,617,2033156
2,ZB,1:남성,2021,01_시즌,봄,니트 셔츠,라운드,봄_니트 셔츠_라운드_ZB,2021-01-03,20076,178239296,1403312400,401,18981424
3,ZA,1:남성,2021,01_시즌,사계절,소품,타이,사계절_소품_타이_ZA,2021-01-03,7652,49967560,381834800,174,8450445
4,ZA,1:남성,2021,01_시즌,봄,수트,블레이져(수트),봄_수트_블레이져(수트)_ZA,2021-01-03,405,26409229,153495000,6,1682760
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51274,ZA,1:남성,2024,01_시즌,사계절,소품,타이,사계절_소품_타이_ZA,2024-12-29,1504,11731200,73696000,4,196000
51275,ZB,1:남성,2024,01_시즌,겨울,수트,수트팬츠,겨울_수트_수트팬츠_ZB,2024-12-29,1404,38437753,209196000,25,2227550
51276,ZB,1:남성,2024,02_이월,여름,스웨터,라운드,여름_스웨터_라운드_ZB,2024-12-29,1926,28828072,190674000,0,0
51277,ZE,1:남성,2024,01_시즌,사계절,우븐 셔츠,드레스셔츠,사계절_우븐 셔츠_드레스셔츠_ZE,2024-12-29,19614,139893880,1567158600,0,0


## 제품 개당 원가/택가/실판매가 컬럼 생성

In [18]:
using_df['제품 개당 원가'] = (using_df['총입고원가'] / using_df['총입고수량']).round(0)
using_df['제품 개당 택가'] = (using_df['총입고택가'] / using_df['총입고수량']).round(0)
using_df['제품 개당 실판매가'] = (using_df['판매액'] / using_df['판매수량']).round(0)
using_df

,라인,성별,기획년도,시즌이월,시즌,복종,소품종,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,제품 개당 원가,제품 개당 택가,제품 개당 실판매가
0,ZB,1:남성,2021,01_시즌,봄,우븐 셔츠,캐쥬얼셔츠,봄_우븐 셔츠_캐쥬얼셔츠_ZB,2021-01-03,2247,19893640,157065300,0,-124850,8853.0,69900.0,-inf
1,ZB,1:남성,2021,01_시즌,사계절,소품,양말,사계절_소품_양말_ZB,2021-01-03,14000,11480000,46200000,617,2033156,820.0,3300.0,3295.0
2,ZB,1:남성,2021,01_시즌,봄,니트 셔츠,라운드,봄_니트 셔츠_라운드_ZB,2021-01-03,20076,178239296,1403312400,401,18981424,8878.0,69900.0,47335.0
3,ZA,1:남성,2021,01_시즌,사계절,소품,타이,사계절_소품_타이_ZA,2021-01-03,7652,49967560,381834800,174,8450445,6530.0,49900.0,48566.0
4,ZA,1:남성,2021,01_시즌,봄,수트,블레이져(수트),봄_수트_블레이져(수트)_ZA,2021-01-03,405,26409229,153495000,6,1682760,65208.0,379000.0,280460.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51274,ZA,1:남성,2024,01_시즌,사계절,소품,타이,사계절_소품_타이_ZA,2024-12-29,1504,11731200,73696000,4,196000,7800.0,49000.0,49000.0
51275,ZB,1:남성,2024,01_시즌,겨울,수트,수트팬츠,겨울_수트_수트팬츠_ZB,2024-12-29,1404,38437753,209196000,25,2227550,27377.0,149000.0,89102.0
51276,ZB,1:남성,2024,02_이월,여름,스웨터,라운드,여름_스웨터_라운드_ZB,2024-12-29,1926,28828072,190674000,0,0,14968.0,99000.0,NaN
51277,ZE,1:남성,2024,01_시즌,사계절,우븐 셔츠,드레스셔츠,사계절_우븐 셔츠_드레스셔츠_ZE,2024-12-29,19614,139893880,1567158600,0,0,7132.0,79900.0,NaN


### inf, nan 값 확인 및 inf를 nan으로 대체

In [19]:
print(using_df['제품 개당 원가'].apply(np.isinf).sum())
print(using_df['제품 개당 택가'].apply(np.isinf).sum())
print(using_df['제품 개당 실판매가'].apply(np.isinf).sum())

0
0
251


In [20]:
print(using_df['제품 개당 원가'].isna().sum())
print(using_df['제품 개당 택가'].isna().sum())
print(using_df['제품 개당 실판매가'].isna().sum())

0
0
7042


In [21]:
using_df['제품 개당 실판매가'].replace([np.inf, -np.inf], np.nan, inplace=True)
using_df['제품 개당 실판매가'].apply(np.isinf).sum()

np.int64(0)

### 실판매가 == 0 or  <0 확인

In [22]:
using_df

,라인,성별,기획년도,시즌이월,시즌,복종,소품종,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,제품 개당 원가,제품 개당 택가,제품 개당 실판매가
0,ZB,1:남성,2021,01_시즌,봄,우븐 셔츠,캐쥬얼셔츠,봄_우븐 셔츠_캐쥬얼셔츠_ZB,2021-01-03,2247,19893640,157065300,0,-124850,8853.0,69900.0,NaN
1,ZB,1:남성,2021,01_시즌,사계절,소품,양말,사계절_소품_양말_ZB,2021-01-03,14000,11480000,46200000,617,2033156,820.0,3300.0,3295.0
2,ZB,1:남성,2021,01_시즌,봄,니트 셔츠,라운드,봄_니트 셔츠_라운드_ZB,2021-01-03,20076,178239296,1403312400,401,18981424,8878.0,69900.0,47335.0
3,ZA,1:남성,2021,01_시즌,사계절,소품,타이,사계절_소품_타이_ZA,2021-01-03,7652,49967560,381834800,174,8450445,6530.0,49900.0,48566.0
4,ZA,1:남성,2021,01_시즌,봄,수트,블레이져(수트),봄_수트_블레이져(수트)_ZA,2021-01-03,405,26409229,153495000,6,1682760,65208.0,379000.0,280460.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51274,ZA,1:남성,2024,01_시즌,사계절,소품,타이,사계절_소품_타이_ZA,2024-12-29,1504,11731200,73696000,4,196000,7800.0,49000.0,49000.0
51275,ZB,1:남성,2024,01_시즌,겨울,수트,수트팬츠,겨울_수트_수트팬츠_ZB,2024-12-29,1404,38437753,209196000,25,2227550,27377.0,149000.0,89102.0
51276,ZB,1:남성,2024,02_이월,여름,스웨터,라운드,여름_스웨터_라운드_ZB,2024-12-29,1926,28828072,190674000,0,0,14968.0,99000.0,NaN
51277,ZE,1:남성,2024,01_시즌,사계절,우븐 셔츠,드레스셔츠,사계절_우븐 셔츠_드레스셔츠_ZE,2024-12-29,19614,139893880,1567158600,0,0,7132.0,79900.0,NaN


In [23]:
using_df[using_df['제품 개당 실판매가'] < 0]

,라인,성별,기획년도,시즌이월,시즌,복종,소품종,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,제품 개당 원가,제품 개당 택가,제품 개당 실판매가
96,ZB,1:남성,2021,01_시즌,봄,우븐 셔츠,캐쥬얼셔츠,봄_우븐 셔츠_캐쥬얼셔츠_ZB,2021-01-10,2995,21348695,209350500,16,-351690,7128.0,69900.0,-21981.0
202,ZB,1:남성,2021,01_시즌,봄,스웨터,오픈형(CARDIGAN),봄_스웨터_오픈형(CARDIGAN)_ZB,2021-01-17,3259,49575364,521114100,44,-5050,15212.0,159900.0,-115.0
231,ZA,1:남성,2021,01_시즌,봄,수트,수트팬츠,봄_수트_수트팬츠_ZA,2021-01-17,489,18401014,107091000,1,-21900,37630.0,219000.0,-21900.0
332,ZB,1:남성,2021,01_시즌,봄,수트,블레이져(수트),봄_수트_블레이져(수트)_ZB,2021-01-31,1516,61021238,301684000,1,-278600,40251.0,199000.0,-278600.0
335,ZB,1:남성,2021,01_시즌,봄,니트 셔츠,라운드,봄_니트 셔츠_라운드_ZB,2021-01-31,27000,169399202,1887300000,73,-7087970,6274.0,69900.0,-97095.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49482,ZB,1:남성,2024,02_이월,여름,스웨터,T-에리,여름_스웨터_T-에리_ZB,2024-12-01,22000,210669018,1958000000,-2,55100,9576.0,89000.0,-27550.0
49500,ZE,1:남성,2024,02_이월,여름,니트 셔츠,라운드,여름_니트 셔츠_라운드_ZE,2024-12-01,16445,97123752,1299155000,-2,14310,5906.0,79000.0,-7155.0
49552,ZE,1:남성,2024,02_이월,여름,니트 셔츠,라운드,여름_니트 셔츠_라운드_ZE,2024-12-01,40424,192786773,3193496000,-1,64997,4769.0,79000.0,-64997.0
49680,ZE,1:남성,2024,02_이월,여름,니트 셔츠,티에리,여름_니트 셔츠_티에리_ZE,2024-12-01,29028,232479138,2293212000,-3,121756,8009.0,79000.0,-40585.0


In [24]:
using_df[using_df['제품 개당 실판매가'] == 0]

,라인,성별,기획년도,시즌이월,시즌,복종,소품종,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,제품 개당 원가,제품 개당 택가,제품 개당 실판매가
4398,ZA,1:남성,2021,02_이월,봄,수트,블레이져(수트),봄_수트_블레이져(수트)_ZA,2021-07-11,1509,78118694,481371000,-1,0,51769.0,319000.0,-0.0
4556,ZA,1:남성,2021,02_이월,봄,수트,수트팬츠,봄_수트_수트팬츠_ZA,2021-07-11,489,18401014,107091000,-1,0,37630.0,219000.0,-0.0
5582,ZD,1:남성,2021,02_이월,봄,우븐 셔츠,캐쥬얼셔츠,봄_우븐 셔츠_캐쥬얼셔츠_ZD,2021-08-15,284,2552194,19851600,1,0,8987.0,69900.0,0.0
7034,ZB,1:남성,2021,01_시즌,여름,팬츠,반바지,여름_팬츠_반바지_ZB,2021-09-19,15204,128114423,910719600,-2,0,8426.0,59900.0,-0.0
7566,ZE,1:남성,2021,02_이월,여름,자켓,싱글재킷,여름_자켓_싱글재킷_ZE,2021-10-03,2697,57943036,536703000,-26,0,21484.0,199000.0,-0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49549,ZE,1:남성,2024,02_이월,여름,자켓,싱글재킷,여름_자켓_싱글재킷_ZE,2024-12-01,6276,169736766,1876524000,-1,0,27045.0,299000.0,-0.0
49553,ZB,1:남성,2024,02_이월,여름,팬츠,팬츠(일반),여름_팬츠_팬츠(일반)_ZB,2024-12-01,16579,201618799,1641321000,-4,0,12161.0,99000.0,-0.0
49677,ZE,1:남성,2024,02_이월,여름,팬츠,팬츠(일반),여름_팬츠_팬츠(일반)_ZE,2024-12-01,2554,28532326,252846000,-1,0,11172.0,99000.0,-0.0
49977,ZB,1:남성,2024,02_이월,여름,팬츠,팬츠(일반),여름_팬츠_팬츠(일반)_ZB,2024-12-08,3947,43506618,390753000,-1,0,11023.0,99000.0,-0.0


## [기획년도,주차,카테고리] 집계 시작 1단계

In [25]:
agg_dict = {
    '판매수량': 'sum',
    '판매액': 'sum',
    '제품 개당 택가': 'mean',
    '제품 개당 원가': 'mean',
}

avg_df_by_group = using_df.groupby(['기획년도', '주차', '카테고리']).agg(agg_dict).reset_index()
avg_df_by_group

,기획년도,주차,카테고리,판매수량,판매액,제품 개당 택가,제품 개당 원가
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,53,52297400,999000.0,204500.000000
1,2021,2021-01-03,봄_니트 셔츠_라운드_ZB,401,18981424,69900.0,8878.000000
2,2021,2021-01-03,봄_니트 셔츠_라운드_ZE,7,214516,69900.0,6179.000000
3,2021,2021-01-03,봄_수트_블레이져(수트)_ZA,109,19685400,339000.0,56150.333333
4,2021,2021-01-03,봄_수트_블레이져(수트)_ZB,194,29890941,199000.0,40583.200000
...,...,...,...,...,...,...,...
20648,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZA,0,0,129000.0,13019.750000
20649,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZB,0,0,99000.0,11930.500000
20650,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZD,0,0,99000.0,13104.000000
20651,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZE,0,0,105000.0,10790.400000


### 중간 확인

In [26]:
avg_df_by_group[avg_df_by_group['카테고리'] == '봄_가죽&FUR_가죽점퍼_ZA']

,기획년도,주차,카테고리,판매수량,판매액,제품 개당 택가,제품 개당 원가
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,53,52297400,999000.0,204500.0
31,2021,2021-01-10,봄_가죽&FUR_가죽점퍼_ZA,108,44542500,999000.0,204500.0
69,2021,2021-01-17,봄_가죽&FUR_가죽점퍼_ZA,79,23101100,999000.0,204500.0
109,2021,2021-01-24,봄_가죽&FUR_가죽점퍼_ZA,51,24770500,999000.0,204500.0
149,2021,2021-01-31,봄_가죽&FUR_가죽점퍼_ZA,170,83960000,999000.0,204500.0
...,...,...,...,...,...,...,...
15152,2023,2023-12-03,봄_가죽&FUR_가죽점퍼_ZA,5,2495000,1149000.0,265036.0
15311,2023,2023-12-10,봄_가죽&FUR_가죽점퍼_ZA,0,0,1149000.0,265036.0
15470,2023,2023-12-17,봄_가죽&FUR_가죽점퍼_ZA,-1,-399000,1149000.0,265036.0
15629,2023,2023-12-24,봄_가죽&FUR_가죽점퍼_ZA,6,3594000,1149000.0,265036.0


## [기획년도,주차,카테고리] 집계 시작 2단계

In [27]:
using_df_filtered = using_df[using_df['제품 개당 실판매가'] > 0]
주차별_실판매가_df = using_df_filtered.groupby(['기획년도', '주차', '카테고리'])['제품 개당 실판매가'].mean().reset_index()

avg_df_by_group = avg_df_by_group.merge(주차별_실판매가_df, on=["기획년도", "주차", "카테고리"], how="left")
avg_df_by_group

,기획년도,주차,카테고리,판매수량,판매액,제품 개당 택가,제품 개당 원가,제품 개당 실판매가
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,53,52297400,999000.0,204500.000000,987705.5
1,2021,2021-01-03,봄_니트 셔츠_라운드_ZB,401,18981424,69900.0,8878.000000,47335.0
2,2021,2021-01-03,봄_니트 셔츠_라운드_ZE,7,214516,69900.0,6179.000000,30645.0
3,2021,2021-01-03,봄_수트_블레이져(수트)_ZA,109,19685400,339000.0,56150.333333,207972.0
4,2021,2021-01-03,봄_수트_블레이져(수트)_ZB,194,29890941,199000.0,40583.200000,199527.8
...,...,...,...,...,...,...,...,...
20648,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZA,0,0,129000.0,13019.750000,NaN
20649,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZB,0,0,99000.0,11930.500000,NaN
20650,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZD,0,0,99000.0,13104.000000,NaN
20651,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZE,0,0,105000.0,10790.400000,NaN


In [28]:
avg_df_by_group.rename(columns={
    "제품 개당 택가": "평균 택가",
    "제품 개당 원가": "평균 원가",
    "제품 개당 실판매가": "주차별 평균 실판매가"
}, inplace=True)
avg_df_by_group

,기획년도,주차,카테고리,판매수량,판매액,평균 택가,평균 원가,주차별 평균 실판매가
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,53,52297400,999000.0,204500.000000,987705.5
1,2021,2021-01-03,봄_니트 셔츠_라운드_ZB,401,18981424,69900.0,8878.000000,47335.0
2,2021,2021-01-03,봄_니트 셔츠_라운드_ZE,7,214516,69900.0,6179.000000,30645.0
3,2021,2021-01-03,봄_수트_블레이져(수트)_ZA,109,19685400,339000.0,56150.333333,207972.0
4,2021,2021-01-03,봄_수트_블레이져(수트)_ZB,194,29890941,199000.0,40583.200000,199527.8
...,...,...,...,...,...,...,...,...
20648,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZA,0,0,129000.0,13019.750000,NaN
20649,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZB,0,0,99000.0,11930.500000,NaN
20650,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZD,0,0,99000.0,13104.000000,NaN
20651,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZE,0,0,105000.0,10790.400000,NaN


### 중간확인

In [29]:
avg_df_by_group[avg_df_by_group['카테고리'] == '봄_가죽&FUR_가죽점퍼_ZA']

,기획년도,주차,카테고리,판매수량,판매액,평균 택가,평균 원가,주차별 평균 실판매가
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,53,52297400,999000.0,204500.0,987705.5
31,2021,2021-01-10,봄_가죽&FUR_가죽점퍼_ZA,108,44542500,999000.0,204500.0,392363.0
69,2021,2021-01-17,봄_가죽&FUR_가죽점퍼_ZA,79,23101100,999000.0,204500.0,294492.0
109,2021,2021-01-24,봄_가죽&FUR_가죽점퍼_ZA,51,24770500,999000.0,204500.0,485429.0
149,2021,2021-01-31,봄_가죽&FUR_가죽점퍼_ZA,170,83960000,999000.0,204500.0,493421.0
...,...,...,...,...,...,...,...,...
15152,2023,2023-12-03,봄_가죽&FUR_가죽점퍼_ZA,5,2495000,1149000.0,265036.0,499000.0
15311,2023,2023-12-10,봄_가죽&FUR_가죽점퍼_ZA,0,0,1149000.0,265036.0,NaN
15470,2023,2023-12-17,봄_가죽&FUR_가죽점퍼_ZA,-1,-399000,1149000.0,265036.0,399000.0
15629,2023,2023-12-24,봄_가죽&FUR_가죽점퍼_ZA,6,3594000,1149000.0,265036.0,599000.0


In [30]:
avg_df_by_group[(avg_df_by_group['카테고리'] == '봄_가죽&FUR_가죽점퍼_ZA') & (avg_df_by_group['판매수량'] >= 0) & (avg_df_by_group['판매액'] < 0)]

,기획년도,주차,카테고리,판매수량,판매액,평균 택가,평균 원가,주차별 평균 실판매가
2220,2021,2021-08-22,봄_가죽&FUR_가죽점퍼_ZA,14,-7346970,999000.0,204500.0,657433.5
3203,2021,2021-10-17,봄_가죽&FUR_가죽점퍼_ZA,17,-1451480,999000.0,204500.0,78752.0
7249,2022,2022-08-14,봄_가죽&FUR_가죽점퍼_ZA,0,-300400,999000.0,183300.0,NaN
13115,2023,2023-09-03,봄_가죽&FUR_가죽점퍼_ZA,1,-1000,1149000.0,265036.0,199000.0


## [기획년도,주차,카테고리] 집계 시작 3단계

In [31]:
using_df_filtered['월'] = using_df_filtered['주차'].dt.month
using_df_filtered.head()

,라인,성별,기획년도,시즌이월,시즌,복종,소품종,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,제품 개당 원가,제품 개당 택가,제품 개당 실판매가,월
1,ZB,1:남성,2021,01_시즌,사계절,소품,양말,사계절_소품_양말_ZB,2021-01-03,14000,11480000,46200000,617,2033156,820.0,3300.0,3295.0,1
2,ZB,1:남성,2021,01_시즌,봄,니트 셔츠,라운드,봄_니트 셔츠_라운드_ZB,2021-01-03,20076,178239296,1403312400,401,18981424,8878.0,69900.0,47335.0,1
3,ZA,1:남성,2021,01_시즌,사계절,소품,타이,사계절_소품_타이_ZA,2021-01-03,7652,49967560,381834800,174,8450445,6530.0,49900.0,48566.0,1
4,ZA,1:남성,2021,01_시즌,봄,수트,블레이져(수트),봄_수트_블레이져(수트)_ZA,2021-01-03,405,26409229,153495000,6,1682760,65208.0,379000.0,280460.0,1
5,ZB,1:남성,2021,01_시즌,봄,수트,수트팬츠,봄_수트_수트팬츠_ZB,2021-01-03,1754,33478840,173646000,24,1682800,19087.0,99000.0,70117.0,1


In [32]:
월별_실판매가_df = using_df_filtered.groupby(['기획년도', '월', '카테고리'])['제품 개당 실판매가'].mean().reset_index()

avg_df_by_group['월'] = avg_df_by_group['주차'].dt.month
avg_df_by_group = avg_df_by_group.merge(월별_실판매가_df, on=['기획년도', '월', '카테고리'], how="left")
avg_df_by_group

,기획년도,주차,카테고리,판매수량,판매액,평균 택가,평균 원가,주차별 평균 실판매가,월,제품 개당 실판매가
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,53,52297400,999000.0,204500.000000,987705.5,1,530682.100000
1,2021,2021-01-03,봄_니트 셔츠_라운드_ZB,401,18981424,69900.0,8878.000000,47335.0,1,41773.500000
2,2021,2021-01-03,봄_니트 셔츠_라운드_ZE,7,214516,69900.0,6179.000000,30645.0,1,24672.600000
3,2021,2021-01-03,봄_수트_블레이져(수트)_ZA,109,19685400,339000.0,56150.333333,207972.0,1,188352.533333
4,2021,2021-01-03,봄_수트_블레이져(수트)_ZB,194,29890941,199000.0,40583.200000,199527.8,1,160160.125000
...,...,...,...,...,...,...,...,...,...,...
20648,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZA,0,0,129000.0,13019.750000,NaN,12,84424.181818
20649,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZB,0,0,99000.0,11930.500000,NaN,12,35369.150000
20650,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZD,0,0,99000.0,13104.000000,NaN,12,NaN
20651,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZE,0,0,105000.0,10790.400000,NaN,12,45195.000000


In [33]:
avg_df_by_group.rename(columns={
    "제품 개당 실판매가": "월별 평균 실판매가"
}, inplace=True)
avg_df_by_group

,기획년도,주차,카테고리,판매수량,판매액,평균 택가,평균 원가,주차별 평균 실판매가,월,월별 평균 실판매가
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,53,52297400,999000.0,204500.000000,987705.5,1,530682.100000
1,2021,2021-01-03,봄_니트 셔츠_라운드_ZB,401,18981424,69900.0,8878.000000,47335.0,1,41773.500000
2,2021,2021-01-03,봄_니트 셔츠_라운드_ZE,7,214516,69900.0,6179.000000,30645.0,1,24672.600000
3,2021,2021-01-03,봄_수트_블레이져(수트)_ZA,109,19685400,339000.0,56150.333333,207972.0,1,188352.533333
4,2021,2021-01-03,봄_수트_블레이져(수트)_ZB,194,29890941,199000.0,40583.200000,199527.8,1,160160.125000
...,...,...,...,...,...,...,...,...,...,...
20648,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZA,0,0,129000.0,13019.750000,NaN,12,84424.181818
20649,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZB,0,0,99000.0,11930.500000,NaN,12,35369.150000
20650,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZD,0,0,99000.0,13104.000000,NaN,12,NaN
20651,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZE,0,0,105000.0,10790.400000,NaN,12,45195.000000


### 중간 확인

In [34]:
avg_df_by_group['월별 평균 실판매가'].isna().sum()

np.int64(2083)

In [35]:
avg_df_by_group['주차별 평균 실판매가'].isna().sum()

np.int64(3574)

## [기획년도,주차,카테고리] 집계 시작 4단계

In [36]:
년도별_실판매가_df = using_df_filtered.groupby(['기획년도', '카테고리'])['제품 개당 실판매가'].mean().reset_index()

avg_df_by_group = avg_df_by_group.merge(년도별_실판매가_df, on=['기획년도', '카테고리'], how="left")
avg_df_by_group

,기획년도,주차,카테고리,판매수량,판매액,평균 택가,평균 원가,주차별 평균 실판매가,월,월별 평균 실판매가,제품 개당 실판매가
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,53,52297400,999000.0,204500.000000,987705.5,1,530682.100000,455749.653061
1,2021,2021-01-03,봄_니트 셔츠_라운드_ZB,401,18981424,69900.0,8878.000000,47335.0,1,41773.500000,24107.081633
2,2021,2021-01-03,봄_니트 셔츠_라운드_ZE,7,214516,69900.0,6179.000000,30645.0,1,24672.600000,16698.758621
3,2021,2021-01-03,봄_수트_블레이져(수트)_ZA,109,19685400,339000.0,56150.333333,207972.0,1,188352.533333,166517.051136
4,2021,2021-01-03,봄_수트_블레이져(수트)_ZB,194,29890941,199000.0,40583.200000,199527.8,1,160160.125000,125210.871245
...,...,...,...,...,...,...,...,...,...,...,...
20648,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZA,0,0,129000.0,13019.750000,NaN,12,84424.181818,77325.703030
20649,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZB,0,0,99000.0,11930.500000,NaN,12,35369.150000,41765.308772
20650,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZD,0,0,99000.0,13104.000000,NaN,12,NaN,26235.000000
20651,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZE,0,0,105000.0,10790.400000,NaN,12,45195.000000,28329.326389


In [37]:
avg_df_by_group.rename(columns={
    "제품 개당 실판매가": "년도별 평균 실판매가"
}, inplace=True)
avg_df_by_group

,기획년도,주차,카테고리,판매수량,판매액,평균 택가,평균 원가,주차별 평균 실판매가,월,월별 평균 실판매가,년도별 평균 실판매가
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,53,52297400,999000.0,204500.000000,987705.5,1,530682.100000,455749.653061
1,2021,2021-01-03,봄_니트 셔츠_라운드_ZB,401,18981424,69900.0,8878.000000,47335.0,1,41773.500000,24107.081633
2,2021,2021-01-03,봄_니트 셔츠_라운드_ZE,7,214516,69900.0,6179.000000,30645.0,1,24672.600000,16698.758621
3,2021,2021-01-03,봄_수트_블레이져(수트)_ZA,109,19685400,339000.0,56150.333333,207972.0,1,188352.533333,166517.051136
4,2021,2021-01-03,봄_수트_블레이져(수트)_ZB,194,29890941,199000.0,40583.200000,199527.8,1,160160.125000,125210.871245
...,...,...,...,...,...,...,...,...,...,...,...
20648,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZA,0,0,129000.0,13019.750000,NaN,12,84424.181818,77325.703030
20649,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZB,0,0,99000.0,11930.500000,NaN,12,35369.150000,41765.308772
20650,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZD,0,0,99000.0,13104.000000,NaN,12,NaN,26235.000000
20651,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZE,0,0,105000.0,10790.400000,NaN,12,45195.000000,28329.326389


### 중간 확인

In [38]:
avg_df_by_group['년도별 평균 실판매가'].isna().sum()

np.int64(52)

In [39]:
avg_df_by_group[avg_df_by_group['년도별 평균 실판매가'].isna()]

,기획년도,주차,카테고리,판매수량,판매액,평균 택가,평균 원가,주차별 평균 실판매가,월,월별 평균 실판매가,년도별 평균 실판매가
4666,2022,2022-01-02,봄_우븐 셔츠_캐쥬얼셔츠_ZD,0,0,129900.0,5221.0,NaN,1,NaN,NaN
4704,2022,2022-01-09,봄_우븐 셔츠_캐쥬얼셔츠_ZD,0,0,129900.0,5221.0,NaN,1,NaN,NaN
4746,2022,2022-01-16,봄_우븐 셔츠_캐쥬얼셔츠_ZD,0,0,129900.0,5221.0,NaN,1,NaN,NaN
4792,2022,2022-01-23,봄_우븐 셔츠_캐쥬얼셔츠_ZD,0,0,129900.0,5221.0,NaN,1,NaN,NaN
4842,2022,2022-01-30,봄_우븐 셔츠_캐쥬얼셔츠_ZD,0,0,129900.0,5221.0,NaN,1,NaN,NaN
4898,2022,2022-02-06,봄_우븐 셔츠_캐쥬얼셔츠_ZD,0,0,129900.0,5221.0,NaN,2,NaN,NaN
4956,2022,2022-02-13,봄_우븐 셔츠_캐쥬얼셔츠_ZD,0,0,129900.0,5221.0,NaN,2,NaN,NaN
5022,2022,2022-02-20,봄_우븐 셔츠_캐쥬얼셔츠_ZD,0,0,129900.0,5221.0,NaN,2,NaN,NaN
5089,2022,2022-02-27,봄_우븐 셔츠_캐쥬얼셔츠_ZD,0,0,129900.0,5221.0,NaN,2,NaN,NaN
5160,2022,2022-03-06,봄_우븐 셔츠_캐쥬얼셔츠_ZD,0,0,129900.0,5221.0,NaN,3,NaN,NaN


## 실판매가 컬럼 추가

In [40]:
avg_df_by_group['실판매가'] = avg_df_by_group['판매액'] / avg_df_by_group['판매수량']
avg_df_by_group

,기획년도,주차,카테고리,판매수량,판매액,평균 택가,평균 원가,주차별 평균 실판매가,월,월별 평균 실판매가,년도별 평균 실판매가,실판매가
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,53,52297400,999000.0,204500.000000,987705.5,1,530682.100000,455749.653061,986743.396226
1,2021,2021-01-03,봄_니트 셔츠_라운드_ZB,401,18981424,69900.0,8878.000000,47335.0,1,41773.500000,24107.081633,47335.221945
2,2021,2021-01-03,봄_니트 셔츠_라운드_ZE,7,214516,69900.0,6179.000000,30645.0,1,24672.600000,16698.758621,30645.142857
3,2021,2021-01-03,봄_수트_블레이져(수트)_ZA,109,19685400,339000.0,56150.333333,207972.0,1,188352.533333,166517.051136,180600.000000
4,2021,2021-01-03,봄_수트_블레이져(수트)_ZB,194,29890941,199000.0,40583.200000,199527.8,1,160160.125000,125210.871245,154077.015464
...,...,...,...,...,...,...,...,...,...,...,...,...
20648,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZA,0,0,129000.0,13019.750000,NaN,12,84424.181818,77325.703030,NaN
20649,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZB,0,0,99000.0,11930.500000,NaN,12,35369.150000,41765.308772,NaN
20650,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZD,0,0,99000.0,13104.000000,NaN,12,NaN,26235.000000,NaN
20651,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZE,0,0,105000.0,10790.400000,NaN,12,45195.000000,28329.326389,NaN


### 중간 확인

In [41]:
avg_df_by_group[avg_df_by_group['실판매가'] <= 0]

,기획년도,주차,카테고리,판매수량,판매액,평균 택가,평균 원가,주차별 평균 실판매가,월,월별 평균 실판매가,년도별 평균 실판매가,실판매가
343,2021,2021-02-21,여름_우븐 셔츠_드레스셔츠_ZB,95,-110210,69900.000000,6329.000000,NaN,2,50254.000000,23706.419048,-1160.105263
407,2021,2021-02-28,여름_점퍼_점퍼_ZE,1,-8875,249000.000000,19370.000000,NaN,2,72441.000000,48061.396552,-8875.000000
974,2021,2021-05-02,봄_조끼_패딩베스트_ZB,26,-352207,79000.000000,10099.000000,NaN,5,11972.250000,16201.333333,-13546.423077
1045,2021,2021-05-09,봄_코트_더블코트_ZB,9,-110449,329000.000000,34815.000000,NaN,5,93452.750000,107073.600000,-12272.111111
1088,2021,2021-05-09,여름_팬츠_반바지_ZB,7,-185152,54900.000000,6646.500000,NaN,5,36940.000000,27559.027397,-26450.285714
...,...,...,...,...,...,...,...,...,...,...,...,...
20083,2024,2024-12-01,여름_우븐 셔츠_캐쥬얼셔츠_ZE,1,0,79000.000000,7492.000000,NaN,12,NaN,21571.290698,0.000000
20094,2024,2024-12-01,여름_팬츠_반바지_ZE,-2,0,79000.000000,7848.000000,NaN,12,NaN,18927.280000,-0.000000
20097,2024,2024-12-01,여름_팬츠_팬츠(일반)_ZB,-2,41838,99000.000000,11930.500000,20919.0,12,35369.150000,41765.308772,-20919.000000
20560,2024,2024-12-29,겨울_자켓_싱글재킷_ZA,16,-4177761,332333.333333,42771.666667,164285.0,12,165827.866667,214754.604167,-261110.062500


In [42]:
avg_df_by_group[avg_df_by_group['주차별 평균 실판매가'] <= 0]

,기획년도,주차,카테고리,판매수량,판매액,평균 택가,평균 원가,주차별 평균 실판매가,월,월별 평균 실판매가,년도별 평균 실판매가,실판매가


In [43]:
print(avg_df_by_group['실판매가'].apply(np.isinf).sum())

91


## 실판매가 이상치 대체

In [44]:
avg_df_by_group['실판매가'].replace([np.inf, -np.inf], np.nan, inplace=True)
avg_df_by_group['실판매가'].apply(np.isinf).sum()

np.int64(0)

In [45]:
avg_df_by_group['실판매가'] = avg_df_by_group['실판매가'].mask(
    (avg_df_by_group['실판매가'] <= 0) | (avg_df_by_group['실판매가'].isna()), 
    avg_df_by_group['주차별 평균 실판매가']
)

avg_df_by_group['실판매가'] = avg_df_by_group['실판매가'].fillna(avg_df_by_group['월별 평균 실판매가'])

avg_df_by_group['실판매가'] = avg_df_by_group['실판매가'].fillna(avg_df_by_group['년도별 평균 실판매가'])

avg_df_by_group

,기획년도,주차,카테고리,판매수량,판매액,평균 택가,평균 원가,주차별 평균 실판매가,월,월별 평균 실판매가,년도별 평균 실판매가,실판매가
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,53,52297400,999000.0,204500.000000,987705.5,1,530682.100000,455749.653061,986743.396226
1,2021,2021-01-03,봄_니트 셔츠_라운드_ZB,401,18981424,69900.0,8878.000000,47335.0,1,41773.500000,24107.081633,47335.221945
2,2021,2021-01-03,봄_니트 셔츠_라운드_ZE,7,214516,69900.0,6179.000000,30645.0,1,24672.600000,16698.758621,30645.142857
3,2021,2021-01-03,봄_수트_블레이져(수트)_ZA,109,19685400,339000.0,56150.333333,207972.0,1,188352.533333,166517.051136,180600.000000
4,2021,2021-01-03,봄_수트_블레이져(수트)_ZB,194,29890941,199000.0,40583.200000,199527.8,1,160160.125000,125210.871245,154077.015464
...,...,...,...,...,...,...,...,...,...,...,...,...
20648,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZA,0,0,129000.0,13019.750000,NaN,12,84424.181818,77325.703030,84424.181818
20649,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZB,0,0,99000.0,11930.500000,NaN,12,35369.150000,41765.308772,35369.150000
20650,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZD,0,0,99000.0,13104.000000,NaN,12,NaN,26235.000000,26235.000000
20651,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZE,0,0,105000.0,10790.400000,NaN,12,45195.000000,28329.326389,45195.000000


### 중간 확인

In [46]:
avg_df_by_group[avg_df_by_group['실판매가'] <= 0]

,기획년도,주차,카테고리,판매수량,판매액,평균 택가,평균 원가,주차별 평균 실판매가,월,월별 평균 실판매가,년도별 평균 실판매가,실판매가


아래 실판매가를 평균값으로 다 대체했는데도 NaN인 경우를 뽑아봤는데, 그냥 쭉~ 판매가 없는 내역인거 같음. 이상없쥬?

In [47]:
avg_df_by_group[avg_df_by_group['실판매가'].isna()]

,기획년도,주차,카테고리,판매수량,판매액,평균 택가,평균 원가,주차별 평균 실판매가,월,월별 평균 실판매가,년도별 평균 실판매가,실판매가
4666,2022,2022-01-02,봄_우븐 셔츠_캐쥬얼셔츠_ZD,0,0,129900.0,5221.0,NaN,1,NaN,NaN,NaN
4704,2022,2022-01-09,봄_우븐 셔츠_캐쥬얼셔츠_ZD,0,0,129900.0,5221.0,NaN,1,NaN,NaN,NaN
4746,2022,2022-01-16,봄_우븐 셔츠_캐쥬얼셔츠_ZD,0,0,129900.0,5221.0,NaN,1,NaN,NaN,NaN
4792,2022,2022-01-23,봄_우븐 셔츠_캐쥬얼셔츠_ZD,0,0,129900.0,5221.0,NaN,1,NaN,NaN,NaN
4842,2022,2022-01-30,봄_우븐 셔츠_캐쥬얼셔츠_ZD,0,0,129900.0,5221.0,NaN,1,NaN,NaN,NaN
4898,2022,2022-02-06,봄_우븐 셔츠_캐쥬얼셔츠_ZD,0,0,129900.0,5221.0,NaN,2,NaN,NaN,NaN
4956,2022,2022-02-13,봄_우븐 셔츠_캐쥬얼셔츠_ZD,0,0,129900.0,5221.0,NaN,2,NaN,NaN,NaN
5022,2022,2022-02-20,봄_우븐 셔츠_캐쥬얼셔츠_ZD,0,0,129900.0,5221.0,NaN,2,NaN,NaN,NaN
5089,2022,2022-02-27,봄_우븐 셔츠_캐쥬얼셔츠_ZD,0,0,129900.0,5221.0,NaN,2,NaN,NaN,NaN
5160,2022,2022-03-06,봄_우븐 셔츠_캐쥬얼셔츠_ZD,0,0,129900.0,5221.0,NaN,3,NaN,NaN,NaN


## 할인율 컬럼 추가

In [48]:
avg_df_by_group['할인율'] = ((avg_df_by_group['평균 택가'] - avg_df_by_group['실판매가']) / avg_df_by_group['평균 택가']) * 100
avg_df_by_group

,기획년도,주차,카테고리,판매수량,판매액,평균 택가,평균 원가,주차별 평균 실판매가,월,월별 평균 실판매가,년도별 평균 실판매가,실판매가,할인율
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,53,52297400,999000.0,204500.000000,987705.5,1,530682.100000,455749.653061,986743.396226,1.226887
1,2021,2021-01-03,봄_니트 셔츠_라운드_ZB,401,18981424,69900.0,8878.000000,47335.0,1,41773.500000,24107.081633,47335.221945,32.281514
2,2021,2021-01-03,봄_니트 셔츠_라운드_ZE,7,214516,69900.0,6179.000000,30645.0,1,24672.600000,16698.758621,30645.142857,56.158594
3,2021,2021-01-03,봄_수트_블레이져(수트)_ZA,109,19685400,339000.0,56150.333333,207972.0,1,188352.533333,166517.051136,180600.000000,46.725664
4,2021,2021-01-03,봄_수트_블레이져(수트)_ZB,194,29890941,199000.0,40583.200000,199527.8,1,160160.125000,125210.871245,154077.015464,22.574364
...,...,...,...,...,...,...,...,...,...,...,...,...,...
20648,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZA,0,0,129000.0,13019.750000,NaN,12,84424.181818,77325.703030,84424.181818,34.554898
20649,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZB,0,0,99000.0,11930.500000,NaN,12,35369.150000,41765.308772,35369.150000,64.273586
20650,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZD,0,0,99000.0,13104.000000,NaN,12,NaN,26235.000000,26235.000000,73.500000
20651,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZE,0,0,105000.0,10790.400000,NaN,12,45195.000000,28329.326389,45195.000000,56.957143


### 중간확인

In [49]:
avg_df_by_group[avg_df_by_group['할인율'] < 0]

,기획년도,주차,카테고리,판매수량,판매액,평균 택가,평균 원가,주차별 평균 실판매가,월,월별 평균 실판매가,년도별 평균 실판매가,실판매가,할인율
23,2021,2021-01-03,사계절_소품_벨트_ZB,184,7443910,39900.0,7683.500000,38215.000000,1,37842.500000,38165.371069,4.045603e+04,-1.393565
55,2021,2021-01-10,봄_코트_싱글코트_ZB,-21,-7550500,249000.0,25439.000000,359548.000000,1,178905.750000,100020.806452,3.595476e+05,-44.396634
91,2021,2021-01-17,봄_코트_더블코트_ZB,-12,-5411000,329000.0,34815.000000,450917.000000,1,243505.000000,107073.600000,4.509167e+05,-37.056738
97,2021,2021-01-17,사계절_소품_벨트_ZB,155,6325950,39900.0,7683.500000,38656.500000,1,37842.500000,38165.371069,4.081258e+04,-2.287170
137,2021,2021-01-24,사계절_소품_벨트_ZB,166,6951880,39900.0,7683.500000,38624.000000,1,37842.500000,38165.371069,4.187880e+04,-4.959386
...,...,...,...,...,...,...,...,...,...,...,...,...,...
19777,2024,2024-11-17,사계절_소품_타이_ZA,1061,52132080,49000.0,7589.655172,49695.482759,11,48610.043103,48250.138745,4.913485e+04,-0.275212
20328,2024,2024-12-15,사계절_소품_양말_ZB,85,487500,5700.0,1192.000000,5700.000000,12,5665.720000,5621.507874,5.735294e+03,-0.619195
20466,2024,2024-12-22,사계절_소품_양말_ZB,100,581590,5700.0,1192.000000,5740.800000,12,5665.720000,5621.507874,5.815900e+03,-2.033333
20523,2024,2024-12-29,가을_우븐 셔츠_캐쥬얼셔츠_ZB,-1,-2179253,79000.0,8819.000000,42441.500000,12,24786.200000,41925.274510,2.179253e+06,-2658.548101


## EDA용 csv 생성

In [50]:
season_df = filtered_df.groupby(['기획년도', '주차', '카테고리'])['시즌이월'].first().reset_index()
season_df

# avg_df_by_group = avg_df_by_group.merge(season_df, on=['기획년도', '주차', '카테고리'], how='left')


,기획년도,주차,카테고리,시즌이월
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,01_시즌
1,2021,2021-01-03,봄_니트 셔츠_라운드_ZB,01_시즌
2,2021,2021-01-03,봄_니트 셔츠_라운드_ZE,01_시즌
3,2021,2021-01-03,봄_수트_블레이져(수트)_ZA,01_시즌
4,2021,2021-01-03,봄_수트_블레이져(수트)_ZB,01_시즌
...,...,...,...,...
21340,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZB,02_이월
21341,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZD,02_이월
21342,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZE,02_이월
21343,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZF,02_이월


In [51]:
avg_df_by_group = avg_df_by_group.merge(season_df, on=['기획년도', '주차', '카테고리'], how='left')
avg_df_by_group

,기획년도,주차,카테고리,판매수량,판매액,평균 택가,평균 원가,주차별 평균 실판매가,월,월별 평균 실판매가,년도별 평균 실판매가,실판매가,할인율,시즌이월
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,53,52297400,999000.0,204500.000000,987705.5,1,530682.100000,455749.653061,986743.396226,1.226887,01_시즌
1,2021,2021-01-03,봄_니트 셔츠_라운드_ZB,401,18981424,69900.0,8878.000000,47335.0,1,41773.500000,24107.081633,47335.221945,32.281514,01_시즌
2,2021,2021-01-03,봄_니트 셔츠_라운드_ZE,7,214516,69900.0,6179.000000,30645.0,1,24672.600000,16698.758621,30645.142857,56.158594,01_시즌
3,2021,2021-01-03,봄_수트_블레이져(수트)_ZA,109,19685400,339000.0,56150.333333,207972.0,1,188352.533333,166517.051136,180600.000000,46.725664,01_시즌
4,2021,2021-01-03,봄_수트_블레이져(수트)_ZB,194,29890941,199000.0,40583.200000,199527.8,1,160160.125000,125210.871245,154077.015464,22.574364,01_시즌
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20648,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZA,0,0,129000.0,13019.750000,NaN,12,84424.181818,77325.703030,84424.181818,34.554898,02_이월
20649,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZB,0,0,99000.0,11930.500000,NaN,12,35369.150000,41765.308772,35369.150000,64.273586,02_이월
20650,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZD,0,0,99000.0,13104.000000,NaN,12,NaN,26235.000000,26235.000000,73.500000,02_이월
20651,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZE,0,0,105000.0,10790.400000,NaN,12,45195.000000,28329.326389,45195.000000,56.957143,02_이월


In [52]:
columns_to_round = ['평균 택가', '평균 원가', '주차별 평균 실판매가', '월별 평균 실판매가', '년도별 평균 실판매가', '실판매가', '할인율']

avg_df_by_group[columns_to_round] = avg_df_by_group[columns_to_round].round(0)

In [53]:
avg_df_by_group.to_csv("EDA_df.csv", index=False, encoding="utf-8-sig")

## 모델링용 전처리(특정 라인(ZD, ZE, ZF) 제거, 시즌이월 '이월' 제거,  '소품', '언더웨어' 제거)

In [54]:
model_df = avg_df_by_group[avg_df_by_group['시즌이월'] != '02_이월']
model_df = model_df[~model_df['카테고리'].str.contains('ZD|ZE|ZF|소품|언더웨어', na=False)]
model_df


,기획년도,주차,카테고리,판매수량,판매액,평균 택가,평균 원가,주차별 평균 실판매가,월,월별 평균 실판매가,년도별 평균 실판매가,실판매가,할인율,시즌이월
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,53,52297400,999000.0,204500.0,987706.0,1,530682.0,455750.0,986743.0,1.0,01_시즌
1,2021,2021-01-03,봄_니트 셔츠_라운드_ZB,401,18981424,69900.0,8878.0,47335.0,1,41774.0,24107.0,47335.0,32.0,01_시즌
3,2021,2021-01-03,봄_수트_블레이져(수트)_ZA,109,19685400,339000.0,56150.0,207972.0,1,188353.0,166517.0,180600.0,47.0,01_시즌
4,2021,2021-01-03,봄_수트_블레이져(수트)_ZB,194,29890941,199000.0,40583.0,199528.0,1,160160.0,125211.0,154077.0,23.0,01_시즌
5,2021,2021-01-03,봄_수트_블레이져(수트)_ZC,33,5652700,239000.0,46064.0,175804.0,1,160403.0,124769.0,171294.0,28.0,01_시즌
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20606,2024,2024-12-29,사계절_수트_블레이져(수트)_ZB,145,17004054,259000.0,42625.0,98988.0,12,105789.0,131261.0,117269.0,55.0,01_시즌
20607,2024,2024-12-29,사계절_수트_블레이져(수트)_ZC,39,5152900,259000.0,44254.0,138896.0,12,140293.0,148324.0,132126.0,49.0,01_시즌
20608,2024,2024-12-29,사계절_수트_수트팬츠_ZB,170,11050814,139000.0,20924.0,56260.0,12,56913.0,69377.0,65005.0,53.0,01_시즌
20609,2024,2024-12-29,사계절_수트_수트팬츠_ZC,51,3838230,139000.0,21438.0,78054.0,12,77179.0,78577.0,75259.0,46.0,01_시즌


In [55]:
model_df.to_csv("model_df.csv", index=False, encoding="utf-8-sig")

## 수량 iqr 이상치 확인

In [56]:
# 판매수량이 음수인 값만 필터링
negative_sales_df = model_df[model_df['판매수량'] < 0]

# IQR 계산
Q1 = negative_sales_df['판매수량'].quantile(0.25)
Q3 = negative_sales_df['판매수량'].quantile(0.75)
IQR = Q3 - Q1

# 3 * IQR 기준으로 이상치 판별
lower_bound = Q1 - 1.5 * IQR

# 이상치 개수 확인
outliers_count = (negative_sales_df['판매수량'] < lower_bound).sum()

# 결과 출력
print(f"이상치 기점: {lower_bound}, 3 IQR 이상인 이상치 개수: {outliers_count}")

이상치 기점: -246.0, 3 IQR 이상인 이상치 개수: 20


In [57]:
model_df[model_df['실판매가']<=0]

,기획년도,주차,카테고리,판매수량,판매액,평균 택가,평균 원가,주차별 평균 실판매가,월,월별 평균 실판매가,년도별 평균 실판매가,실판매가,할인율,시즌이월


In [58]:
model_df[model_df['실판매가'].isna()]

,기획년도,주차,카테고리,판매수량,판매액,평균 택가,평균 원가,주차별 평균 실판매가,월,월별 평균 실판매가,년도별 평균 실판매가,실판매가,할인율,시즌이월


In [59]:
model_df[model_df['할인율']<=0]

,기획년도,주차,카테고리,판매수량,판매액,평균 택가,평균 원가,주차별 평균 실판매가,월,월별 평균 실판매가,년도별 평균 실판매가,실판매가,할인율,시즌이월
20,2021,2021-01-03,봄_코트_더블코트_ZB,45,14805000,329000.0,34814.0,329000.0,1,243505.0,107074.0,329000.0,0.0,01_시즌
22,2021,2021-01-03,봄_코트_싱글코트_ZB,79,19646100,249000.0,25314.0,248685.0,1,178906.0,100021.0,248685.0,0.0,01_시즌
55,2021,2021-01-10,봄_코트_싱글코트_ZB,-21,-7550500,249000.0,25439.0,359548.0,1,178906.0,100021.0,359548.0,-44.0,01_시즌
91,2021,2021-01-17,봄_코트_더블코트_ZB,-12,-5411000,329000.0,34815.0,450917.0,1,243505.0,107074.0,450917.0,-37.0,01_시즌
105,2021,2021-01-17,사계절_자켓_싱글재킷_ZB,0,0,199000.0,23618.0,NaN,1,199000.0,110719.0,199000.0,0.0,01_시즌
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18389,2024,2024-09-08,겨울_점퍼_패딩점퍼_ZB,-2,-448200,224000.0,32296.0,224100.0,9,180525.0,139210.0,224100.0,-0.0,01_시즌
18719,2024,2024-09-22,여름_점퍼_점퍼_ZB,2,528000,249000.0,24849.0,100209.0,9,101424.0,108673.0,264000.0,-6.0,01_시즌
18754,2024,2024-09-29,겨울_사파리_패딩사파리_ZB,-3,-1197000,399000.0,25274.0,399000.0,9,399000.0,201703.0,399000.0,0.0,01_시즌
18900,2024,2024-10-06,겨울_스웨터_Turtle_ZB,80,9714900,119000.0,15076.0,119937.0,10,97449.0,68760.0,121436.0,-2.0,01_시즌


In [60]:
model_df[model_df['할인율'].isna()]

,기획년도,주차,카테고리,판매수량,판매액,평균 택가,평균 원가,주차별 평균 실판매가,월,월별 평균 실판매가,년도별 평균 실판매가,실판매가,할인율,시즌이월


In [61]:
def update_data(df, input_rows, output_rows):
    df = df.copy()  # 원본 데이터 보호

    # Step 1: output 행 값 변경 (input과 output을 합쳐서 덮어쓰기)
    for i in range(len(input_rows)):
        input_date, category = input_rows[i]
        output_date, _ = output_rows[i]

        # input과 output 행 찾기
        mask_input = (df['주차'] == input_date) & (df['카테고리'] == category)
        mask_output = (df['주차'] == output_date) & (df['카테고리'] == category)

        # input 행과 output 행 판매수량 합치기
        df.loc[mask_output, '판매수량'] = (
            df.loc[mask_input, '판매수량'].values +
            df.loc[mask_output, '판매수량'].values
        )
        
        # 판매액, 매출원가, 판매택가 재계산
        df.loc[mask_output, '판매액'] = (df.loc[mask_output, '판매수량'] * df.loc[mask_output, '주차별 평균 실판매가']).round(0)
        df.loc[mask_output, '매출원가계'] = (df.loc[mask_output, '판매수량'] * df.loc[mask_output, '평균 원가']).round(0)
        df.loc[mask_output, '판매택가계'] = (df.loc[mask_output, '판매수량'] * df.loc[mask_output, '평균 택가']).round(0)
        
        # 제품실판가, 할인율 재계산
        df.loc[mask_output, '실판매가'] = df.loc[mask_output, '판매액'] / df.loc[mask_output, '판매수량'] #null 값 있음
        df.loc[mask_output, '할인율'] = (df.loc[mask_output, '평균 택가'] - df.loc[mask_output, '실판매가']) * 100 / df.loc[mask_output, '평균 택가'] #null 값 있음
        
    # Step 2: input 행 값 변경 (이전, 이후 주차 평균으로 대체) -> 0으로 변경
    for input_date, category in input_rows:
        mask_input = (df['주차'] == input_date) & (df['카테고리'] == category)

        # input 행 값 변경
        df.loc[mask_input, ['판매액', '판매수량', '매출원가계', '판매택가계']] = 0

        # 제품실판가,할인율
        # 제품실판가, 할인율을 NaN으로 설정
        df.loc[mask_input, ['실판매가', '할인율']] = np.nan
        
    # 누적 집계값 업데이트
    df['누적판매수량'] = df.groupby(['기획년도', '카테고리'])['판매수량'].cumsum()
    df['누적판매액'] = df.groupby(['기획년도','카테고리'])['판매액'].cumsum()
    df['누적매출원가'] = df.groupby(['기획년도','카테고리'])['매출원가계'].cumsum()
    df['누적판매택가'] = df.groupby(['기획년도','카테고리'])['판매택가계'].cumsum()
        
    # 판매율 / ROI 업데이트
    # df['누적판매율(%)'] = (df['누적판매수량'] / df['총입고수량']* 100).round(2)
    # df['ROI'] = ((df['누적판매액']/1.1 - df['누적매출원가'])/df['총입고원가']).round(2)
        
    return df

# 실행
input_rows = [
    ('2022-09-18', '겨울_스웨터_라운드_ZB'),
    ('2024-04-07', '봄_자켓_싱글재킷_ZB'),
    ('2022-07-31', '여름_니트 셔츠_라운드_ZB'),
    ('2022-09-25', '여름_니트 셔츠_라운드_ZB'),
    ('2022-07-24', '여름_자켓_싱글재킷_ZB'),
    ('2022-08-07', '여름_자켓_싱글재킷_ZB'),
    ('2022-12-11', '사계절_우븐 셔츠_드레스셔츠_ZB'),
    ('2023-10-08', '사계절_데님_데님팬츠_ZB'),
    ('2023-10-15', '사계절_데님_데님팬츠_ZB'),
    ('2023-10-29', '사계절_데님_데님팬츠_ZB'),
    ('2023-12-10', '사계절_데님_데님팬츠_ZB'),
    ('2024-12-29', '사계절_데님_데님팬츠_ZB'),
    ('2023-07-09', '사계절_니트 셔츠_라운드_ZB'),
    ('2023-07-16', '사계절_니트 셔츠_라운드_ZB'),
    ('2023-07-23', '사계절_니트 셔츠_라운드_ZB'),
    ('2023-08-20', '사계절_니트 셔츠_라운드_ZB'),
    ('2023-09-24', '사계절_니트 셔츠_라운드_ZB'),
    ('2023-12-10', '사계절_니트 셔츠_라운드_ZB')
]

output_rows = [
    ('2022-09-11', '겨울_스웨터_라운드_ZB'),
    ('2024-03-31', '봄_자켓_싱글재킷_ZB'),
    ('2022-07-10', '여름_니트 셔츠_라운드_ZB'),
    ('2022-09-18', '여름_니트 셔츠_라운드_ZB'),
    ('2022-07-10', '여름_자켓_싱글재킷_ZB'),
    ('2022-07-31', '여름_자켓_싱글재킷_ZB'),
    ('2022-12-04', '사계절_우븐 셔츠_드레스셔츠_ZB'),
    ('2023-10-01', '사계절_데님_데님팬츠_ZB'),
    ('2023-10-01', '사계절_데님_데님팬츠_ZB'),
    ('2023-10-22', '사계절_데님_데님팬츠_ZB'),
    ('2023-12-03', '사계절_데님_데님팬츠_ZB'),
    ('2024-12-22', '사계절_데님_데님팬츠_ZB'),
    ('2023-06-18', '사계절_니트 셔츠_라운드_ZB'),
    ('2023-07-02', '사계절_니트 셔츠_라운드_ZB'),
    ('2023-06-18', '사계절_니트 셔츠_라운드_ZB'),
    ('2023-08-13', '사계절_니트 셔츠_라운드_ZB'),
    ('2023-09-03', '사계절_니트 셔츠_라운드_ZB'),
    ('2023-12-03', '사계절_니트 셔츠_라운드_ZB')
]

df_updated = update_data(model_df, input_rows, output_rows)